In [1]:
%run Packages_and_Functions.ipynb

import pandas as pd
import seaborn as sns

In [2]:
def random_weighted_graph_uniform(n, p, lower_weight = .01, upper_weight = 1): 
    # Does not take metrics into account, just a regular random graph
    g = graphs.RandomGNP(n,p)
    m = g.num_edges()
    weights = np.random.uniform(size=m)
    uw_edges = g.edges(sort = True)
    
    # Create weighted graph edge list 
    w_edges = [(uw_edges[i][0], uw_edges[i][1], weights[i]) for i in range(m)]
    
    return Graph(w_edges, weighted = True)

def random_weighted_graph_exponential(n, p): 
    # Does not take metrics into account, just a regular random graph
    g = graphs.RandomGNP(n,p)
    m = g.num_edges()
    weights = np.random.exponential(size=m)
    uw_edges = g.edges(sort = True)
    
    # Create weighted graph edge list 
    w_edges = [(uw_edges[i][0], uw_edges[i][1], weights[i]) for i in range(m)]
    
    return Graph(w_edges, weighted = True)

In [3]:
def csv_to_array(path):
    df = pd.read_csv(path,index_col=0)  
    return df.to_numpy()

In [4]:
def is_broken(K):
    A = K.weighted_adjacency_matrix()
    n = len(A[0])
    for i in range(n):
        for j in range(n):
            for k in range(j):
                if A[i,k] > A[i,j] + A[j,k]:
                    return 1
    return 0
    
def has_cycle(G):
    for u,v in G.edges(sort=True,labels=False):
        H =G.copy()
        H.delete_edge(u,v)
        if len(H.shortest_path(u,v)) > 0:
            return 1
    return 0


def experiment_demonstrate_threshold_prob(N = 40,AVG = 100, ps=100,std=False):
    delta = 1.9/ps
    P = [2 - i*delta for i in range(ps)]
    res = np.zeros((2,ps))
    res_std = np.zeros((2,ps))
    for j,q in enumerate(P):
        print(f"finished {(j/ps)*100}%")
        p = 1/np.power(N,q)
        samples_broken=np.zeros(AVG)
        samples_cycles=np.zeros(AVG)
        has_broken = 0
        has_a_cycle = 0
        for trial in range(AVG):
            G = random_weighted_graph_uniform(N,p)
            while G.size() == 0: # Make sure the graph has edges
                G = random_weighted_graph_uniform(N,p)
            bit = has_cycle(G)
            samples_cycles[trial]= bit
            has_a_cycle +=bit    

            bit = 0
            for CC in connected_components_subgraphs(G):
                bit = is_broken(complete(CC))
                if bit == 1:
                    samples_broken[trial] = bit
                    has_broken +=bit
                    break
        res[0,j] = has_broken/AVG ### Broken Cycles
        res[1,j] = has_a_cycle/AVG # how many induced cycles 
        res_std[0,j]=np.std(samples_broken)/np.sqrt(AVG)
        res_std[1,j]=np.std(samples_cycles)/np.sqrt(AVG)         
    if std:
        return res,res_std
    else:
        return res
    


### Experiment - how many edges we discard when running algorithms
def experiment_heuristics_on_G_and_KG(n1=50,n2=130,p = lambda x: np.power(x,-0.5),AVG = 80):
    """
    p would be a function of n, or a constant function. But we definitly expect it to be a function N -> (0,1)
    """
    N = range(n1,n2)
    res = np.zeros((11,len(N)))
    start_time = time.time()
    for i,n in enumerate(N):
        ### Keep track of progress
        if i % 10 == 0:
            print(f"Completed: {i/len(N)*100}% \n" + "Elapsed Time: " + str((time.time() - start_time)/60))
        ### Iterate over connected Components
        for j in range(AVG):
            B = False
            while B == False: # Make sure the graph is broken
                G = random_weighted_graph_uniform(n,p(n))
                DOMR_solns = list()
                for CC in connected_components_subgraphs(G):
                    DOMR = domr_alg(CC)
                    if len(DOMR) > 0:
                        B = True
                    DOMR_solns.append(len(DOMR))
                if B:
                    PIVOT_solns = list()
                    LHS_solns = list()
                    SPC_MR_solns = list()
                    SPC_IOMR_solns = list()
                    REDUCED_SPC_MR_solns = list() # running SPC on complete and discarding whatever's not needed
                    REDUCED_SPC_IOMR_solns = list()
                    
                    COM_PIVOT_solns = list()
                    COM_LHS_solns = list()
                    COM_SPC_MR_solns = list()
                    COM_SPC_IOMR_solns = list()
                    
                    REDUCED_SPC_MR_solns = list() # running SPC on complete and discarding whatever's not needed
                    REDUCED_SPC_IOMR_solns = list()
                    for CC in connected_components_subgraphs(G):
                        comCC = complete(CC)
                        PIVOT_solns.append(len(pivot_heuristic(CC)))
                        SPC_MR_solns.append(len(shortest_path_cover(CC)))
                        SPC_IOMR_solns.append(len(shortest_path_cover(CC,general=0)))

                        LHS_solns.append(len(left_edge_heuristic(CC)))
                        COM_PIVOT_solns.append(len(pivot_heuristic(comCC)))
                        COM_LHS_solns.append(len(left_edge_heuristic(comCC)))

                        # save the result for completion, need to reduce    
#                         complete_MR_SPC = shortest_path_cover(comCC) 
#                         complete_IOMR_SPC = shortest_path_cover(comCC,general=0)

#                         COM_SPC_MR_solns.append(len(complete_MR_SPC))
#                         COM_SPC_IOMR_solns.append(len(complete_IOMR_SPC))

#                         REDUCED_SPC_MR_solns.append(len(reduce_solution(complete_MR_SPC,CC)))
#                         REDUCED_SPC_IOMR_solns.append(len(reduce_solution(complete_IOMR_SPC,CC)))

                    res[0,i] += sum(DOMR_solns)/AVG
                    res[1,i] += sum(PIVOT_solns)/AVG
                    res[2,i] += sum(COM_PIVOT_solns)/AVG
                    res[3,i] += sum(LHS_solns)/AVG
                    res[4,i] += sum(COM_LHS_solns)/AVG
                    res[5,i] += sum(SPC_MR_solns)/AVG
#                     res[6,i] += sum(COM_SPC_MR_solns)/AVG
                    res[7,i] += sum(SPC_IOMR_solns)/AVG
#                     res[8,i] += sum(COM_SPC_IOMR_solns)/AVG
#                     res[9,i] += sum(REDUCED_SPC_MR_solns)/AVG
#                     res[10,i] += sum(REDUCED_SPC_IOMR_solns)/AVG

    return res

def experiment_algs_as_function_of_p(start=1-0.1, end=0.1,ps=100,AVG = 100,n=70):
    delta = (start-end)/ps
    P = [end + delta*i for i in range(ps)]
    res = res = np.zeros((11,len(P)))
    start_time = time.time()

    for i,p in enumerate(P):
        if i % 10 == 0:
            print(f"Completed: {i/len(P)*100}% \n" + "Elapsed Time: " + str((time.time() - start_time)/60))
        for j in range(AVG):
            B = False
            while B == False: # Make sure the graph is broken
                G = random_weighted_graph_uniform(n,p)
                DOMR_solns = list()
                for CC in connected_components_subgraphs(G):
                    DOMR = domr_alg(CC)
                    if len(DOMR) > 0:
                        B = True
                    DOMR_solns.append(len(DOMR))
                if B:
                    PIVOT_solns = list()
                    LHS_solns = list()
                    SPC_MR_solns = list()
                    SPC_IOMR_solns = list()
                    REDUCED_SPC_MR_solns = list() # running SPC on complete and discarding whatever's not needed
                    REDUCED_SPC_IOMR_solns = list()
                    
                    COM_PIVOT_solns = list()
                    COM_LHS_solns = list()
                    COM_SPC_MR_solns = list()
                    COM_SPC_IOMR_solns = list()
                    
                    REDUCED_SPC_MR_solns = list() # running SPC on complete and discarding whatever's not needed
                    REDUCED_SPC_IOMR_solns = list()
                    
                    for CC in connected_components_subgraphs(G):
                        comCC = complete(CC)
                        PIVOT_solns.append(len(pivot_heuristic(CC)))
                        SPC_MR_solns.append(len(shortest_path_cover(CC)))
                        SPC_IOMR_solns.append(len(shortest_path_cover(CC,general=0)))

                        LHS_solns.append(len(left_edge_heuristic(CC)))
                        COM_PIVOT_solns.append(len(pivot_heuristic(comCC)))
                        COM_LHS_solns.append(len(left_edge_heuristic(comCC)))

                        # save the result for completion, need to reduce    
#                         complete_MR_SPC = shortest_path_cover(comCC) 
#                         complete_IOMR_SPC = shortest_path_cover(comCC,general=0)

#                         COM_SPC_MR_solns.append(len(complete_MR_SPC))
#                         COM_SPC_IOMR_solns.append(len(complete_IOMR_SPC))

#                         REDUCED_SPC_MR_solns.append(len(reduce_solution(complete_MR_SPC,CC)))
#                         REDUCED_SPC_IOMR_solns.append(len(reduce_solution(complete_IOMR_SPC,CC)))

                    res[0,i] += sum(DOMR_solns)/AVG
                    res[1,i] += sum(PIVOT_solns)/AVG
                    res[2,i] += sum(COM_PIVOT_solns)/AVG
                    res[3,i] += sum(LHS_solns)/AVG
                    res[4,i] += sum(COM_LHS_solns)/AVG
                    res[5,i] += sum(SPC_MR_solns)/AVG
#                     res[6,i] += sum(COM_SPC_MR_solns)/AVG
                    res[7,i] += sum(SPC_IOMR_solns)/AVG
#                     res[8,i] += sum(COM_SPC_IOMR_solns)/AVG
#                     res[9,i] += sum(REDUCED_SPC_MR_solns)/AVG
#                     res[10,i] += sum(REDUCED_SPC_IOMR_solns)/AVG
    return res

In [6]:
# n1 = 20
# n2 = 50
# AVG =30
# ps = 20

In [12]:
# print("###############################################")
# print("Starting Experiments")
# print("###############################################\n")
# print("Starting: Threshold phenomena")
# print("####################\n")

# ar_df,ar_dfstd = experiment_demonstrate_threshold_prob(AVG=AVG,ps=ps,std=True)
# df,dfstd = pd.DataFrame(ar_df), pd.DataFrame(ar_dfstd)
# df.to_csv("expon_dem_threshold_prob.csv")
# dfstd.to_csv("expon_dem_threshold_prob_std.csv")



###############################################
Starting Experiments
###############################################

Starting: Threshold phenomena
####################

finished 0%
finished 5%
finished 10%
finished 15%
finished 20%
finished 25%
finished 30%
finished 35%
finished 40%
finished 45%
finished 50%
finished 55%
finished 60%
finished 65%
finished 70%
finished 75%
finished 80%
finished 85%
finished 90%
finished 95%


In [13]:
# print("\n###############################################")
# print("Starting: Algorithms\n")
# print("For Gamma(n,p)")
# print("p=0.8")
# df = pd.DataFrame(experiment_heuristics_on_G_and_KG(p=lambda x: .8))
# df.to_csv("expon_alg_p08.csv")

# print("####################\n")
# print("p=0.2")
# df = pd.DataFrame(experiment_heuristics_on_G_and_KG(p= lambda x: .3))
# df.to_csv("expon_alg_p02.csv")


# print("####################\n")
# print("p=1/sqrt(n) - to make sure we have lots of broken cycles") ## To make sure it's connected
# df = pd.DataFrame(experiment_heuristics_on_G_and_KG(p= lambda x: (1)/np.power(x,.5)))
# df.to_csv("expon_alg_p_sqrt(n).csv")

# print("####################\n")
# print("p=2/n - giant CC")
# df = pd.DataFrame(experiment_heuristics_on_G_and_KG(p= lambda x: 2/x))
# df.to_csv("expon_alg_p1overn.csv")


###############################################
Starting: Algorithms

For Gamma(n,p)
p=0.8
Completed: 0.0% 
Elapsed Time: 1.0291735331217447e-06
Completed: 12.5% 
Elapsed Time: 14.985084148248037
Completed: 25.0% 
Elapsed Time: 42.37695709864298
Completed: 37.5% 
Elapsed Time: 104.3057477315267
Completed: 50.0% 
Elapsed Time: 370.4205095132192
Completed: 62.5% 
Elapsed Time: 545.6090067307155
Completed: 75.0% 
Elapsed Time: 757.7818281491598
Completed: 87.5% 
Elapsed Time: 1136.9712511817613
####################

p=0.2
Completed: 0.0% 
Elapsed Time: 3.4968058268229164e-07
Completed: 12.5% 
Elapsed Time: 7.946679035822551
Completed: 25.0% 
Elapsed Time: 21.441395699977875
Completed: 37.5% 
Elapsed Time: 42.93209068377813
Completed: 50.0% 
Elapsed Time: 74.4384606162707
Completed: 62.5% 
Elapsed Time: 119.91755070288976
Completed: 75.0% 
Elapsed Time: 183.36825931866963
Completed: 87.5% 
Elapsed Time: 263.93333096901574
####################

p=1/sqrt(n) - to make sure we have lots of br

In [14]:
# print("\n###############################################")
# print("Starting: Algorithms as function of p\n")
# df = pd.DataFrame(experiment_algs_as_function_of_p())
# df.to_csv("expon_algos_as_fn_p.csv")


###############################################
Starting: Algorithms as function of p

Completed: 0.0% 
Elapsed Time: 2.86102294921875e-07
Completed: 10.0% 
Elapsed Time: 13.77903130054474
Completed: 20.0% 
Elapsed Time: 29.882475050290427
Completed: 30.0% 
Elapsed Time: 48.539272487163544
Completed: 40.0% 
Elapsed Time: 70.4680823524793
Completed: 50.0% 
Elapsed Time: 95.5338942170143
Completed: 60.0% 
Elapsed Time: 124.79942296743393
Completed: 70.0% 
Elapsed Time: 158.72371558348337
Completed: 80.0% 
Elapsed Time: 198.15887331962585
Completed: 90.0% 
Elapsed Time: 242.7217333038648
